In [ ]:
import transformers
import torch
import datasets
import pandas as pd
from sklearn.model_selection import train_test_split
print(torch.cuda.is_available())  # Should return True if GPU is ready

In [ ]:
train = pd.read_csv("train_cleaned2.csv")
test = pd.read_csv("test_cleaned2.csv")

In [ ]:
train_df, val_df = train_test_split(train, test_size=0.2, random_state=42, stratify=train['label'])
train_df.to_csv("train_df_cleaned.csv", index=False)
val_df.to_csv("val_df_cleaned.csv", index=False)

data_files = {"train": "train_df_cleaned.csv", "test": "val_df_cleaned.csv"}
dataset = datasets.load_dataset("csv", data_files=data_files)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "cahya/roberta-base-indonesian-522M"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

print(tokenized_datasets['train'][0])

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = dataset["train"]["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
print(class_weights)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8)

print(model.config)

In [ ]:
label_mapping = {'ideologi': 0, 'pertahanan': 1, 'reformasi': 2, 'harmoni': 3, 'sdm': 4, 'pekerjaan': 5, 'pemerataan': 6, 'hilirisasi': 7}
dataset = dataset.map(lambda x: {"label": label_mapping[x["label"]]})

print(dataset["train"][0])

In [ ]:
# Freeze all layers except the classifier
for param in model.roberta.parameters():
    param.requires_grad = False

# Keep only the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

In [ ]:
import torch.nn as nn

class CustomBERTModel(nn.Module):
    def __init__(self, pretrained_model_name, num_labels):
        super(CustomBERTModel, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name, num_labels=num_labels)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # pooled_output = self.dropout(output[1])  # Applying dropout
        logits = self.fc(output)  # Adding a fully connected layer
        return logits

# Initialize the custom model
custom_model = CustomBERTModel(model_name, num_labels=8)

In [ ]:
from transformers import TrainingArguments, IntervalStrategy

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",           # Directory for saving model checkpoints
    eval_strategy=IntervalStrategy.EPOCH,     # Evaluate at the end of each epoch
    save_strategy="epoch",           # Save at the end of each epoch to match evaluation
    # learning_rate=5e-5,              # Start with a small learning rate
    per_device_train_batch_size=8,  # Batch size per GPU
    per_device_eval_batch_size=8,   # Batch size for evaluation
    num_train_epochs=50,              # Number of epochs
    # weight_decay=0.01,               # Regularization
    save_total_limit=2,              # Limit checkpoints to save space
    load_best_model_at_end=True,     # Automatically load the best checkpoint
    logging_dir="./logs",            # Directory for logs
    logging_steps=100,               # Log every 100 steps
    fp16=True                        # Enable mixed precision for faster training
)

print(training_args)

In [ ]:
from transformers import Trainer
from evaluate import load

# Load a metric (F1-score in this case)
metric = load("accuracy")

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Fix the label mapping issue - ensure labels are properly converted to integers
def map_labels(example):
    if isinstance(example['label'], str):
        return {'label': label_mapping[example['label']]}
    return example

# Apply the mapping again to ensure proper conversion
tokenized_datasets = tokenized_datasets.map(map_labels)

# Verify the label format
print("Sample labels:", tokenized_datasets["train"]["label"][:15])
print("Label type:", type(tokenized_datasets["train"]["label"][0]))

trainer = Trainer(
    model=model,                        # Pre-trained BERT model
    args=training_args,                 # Training arguments
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,        # Efficient batching
    compute_metrics=compute_metrics     # Custom metric
)

: 

In [ ]:
# Start training
trainer.train()